# Experimento A/B en página de inicio

El objetivo de este proyecto es evaluar un **experimento A/B** realizado en una página de inicio (landing page) con versiónes **A y B** para apoyar una **decisión de negocio basada en datos**.

---

El archivo `landing_experiment.csv` contiene información de usuarios expuestos a dos versiones de la página de inicio (landing page) dentro del experimento A/B. Incluye región, dispositivo, fuente de tráfico, tipo de usuario, conversión y gasto.

El análisis sigue una lógica clara y progresiva:

1. 🔍 Explorar y validar los datos.

2. 💰 Comparar el **gasto promedio** por usuario entre la página A y B.

3. 🎯 Comparar la **tasa de conversión** entre la página A y B.

4. 🌐 Revisar **la relación entre la fuente de tráfico y la conversión**.

5. 👤 Revisar **la relación entre el tipo de usuario y la conversión**.

6. 📈 **Visualizar los resultados**: Respalda tus conclusiones mediante gráficos claros.

Se aplican **puebas estadísticas apropiados** para comparar las páginas y **recomendar qué versión es mejor**, justificando la decisión con datos.


## 🧩 Paso 1: Cargar y validar los datos

### 1.1 Carga de datos y vista rápida

In [1]:
# importar librerías
import pandas as pd
from scipy.stats import ttest_ind
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import chi2_contingency
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# cargar archivo
df = pd.read_csv('/datasets/landing_experiment.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/datasets/landing_experiment.csv'

**Vista previa e información general del conjunto de datos**

In [ ]:
# mostrar las primeras 5 filas
df.head()

In [ ]:
df.shape

In [ ]:
# información general
df.info()

✍️ **Comentario**:
    - La columna date es de tipo de dato object y debería de ser fecha .
    - El dataset no tiene valores ausentes.


In [ ]:
#limpieza de la columna date
df['date']= pd.to_datetime(df['date'], errors='coerce')

In [ ]:
#revisión de la conversión a datetime
df.info()

**Descripción del conjunto de datos**

El dataset contiene las siguientes columnas:

- `user_id` — Identificador único del usuario
- `date` — Fecha en la que el usuario fue expuesto a la página
- `landing` — Versión de la página mostrada al usuario
- `region` — Región geográfica del usuario
- `dispositivo` — Tipo de dispositivo utilizado por el usuario
- `traffic_source` — Canal por el que llegó el usuario
- `user_type` — Tipo de usuario según historial previo
- `converted` — Indica si el usuario realizó una conversión
- `gasto` — Monto gastado por el usuario (0 si no convirtió)

### 1.2 Análisis exploratorio y revisión de calidad de datos

Se identifican las variables clave del experimento A/B y se valida que estén bien definidas, completas y que sean consistentes.


 **Variable `user_id`**  
 Verificar usuarios únicos

In [ ]:
df['user_id'].value_counts()

 **Variable `date`**  
Explorar rango de fechas

In [ ]:
# Resumen estadístico
df["date"].describe()

In [ ]:
# Identificar rango temporal del experimento
print("Fecha mínima:", df["date"].min())
print("Fecha máxima:", df["date"].max())

**Variable `gasto` (numérica)**

In [ ]:
# Resumen estadístico
df['gasto'].describe()

In [ ]:
# Resumen estadístico de usuarios que se convirtieron
df['converted'].describe()

 **Variables categóricas**  
 Verificar categorías esperadas del experimento ( A y B).

In [ ]:
# Explorar variables categóricas y cómo se distribuyen
columnas_categoricas= df[['landing','region','dispositivo','traffic_source','user_type']]
for col in columnas_categoricas:
    print("\nConteo de categorías:")
    print(df[col].value_counts())



✍️ **Comentario**:
    - Todas las columnas tienen valores esperados.

## 💰 Paso 2: Comparar el gasto promedio por usuario (página A vs B)

Se evalua si existen diferencias estadísticamente significativas en el gasto promedio de los **usuarios que se convirtieron en clientes** entre la página A y la página B, para identificar qué versión genera **mayor valor económico** para el negocio.


In [ ]:
# Gasto por versión
gasto_A = df[(df['landing']== 'A') & (df['converted'] == 1)]['gasto']
gasto_B = df[(df['landing']== 'B') & (df['converted'] == 1)]['gasto']

# Verificar cantidad de datos que tiene cada grupo
len(gasto_A), len(gasto_B)

### Prueba T de student

**Hipótesis:**
- **Hipótesis nula (H₀):** El gasto promedio es igual en ambas páginas
- **Hipótesis alternativa (H₁):** el gasto promedio es diferente en ambas páginas

In [ ]:
# Aplicar prueba
t_stat, p_value= ttest_ind(gasto_A, gasto_B)


# Visualizar resultados
print(f"Estadístico : {t_stat}")
print(f"Valor p: {p_value}")

In [ ]:
alpha= 0.05
if p_value < alpha:
    print('Rechazamos la hipótesis nula, hay evidencia de una diferencia')
else:
    print('No rechazamos la hipótesis nula, no hay evidencia de una diferencia')

In [ ]:
promedio_A = gasto_A.mean()
promedio_B = gasto_B.mean()
print('El promedio de gasto del grupo A es de :', promedio_A)
print('El promedio de gasto del grupo B es de :', promedio_B)
print('La diferencia es de :', promedio_B - promedio_A)

### 📝 Conclusión e interpretación

**Decisión:**  
Rechazamos la hipótesis nula, hay una diferencia estadísticamente significativa entre el gasto promedio de la página A y B.

**Interpretación de negocio:**  
El grupo B tuvo un gasto promedio mas alto que el grupo A.
Este resultado se basa en una muestra de usuarios que realizaron una compra y no evalúa si la diferencia observada es suficiente para justificar un cambio desde una perspectiva de negocio.


---


## 📈 Paso 3: Comparar la tasa de conversión entre la página A y B

Se evalua si existen difere]ncias estadísticamente significativas en la **tasa de conversión** entre la página A y B, con el fin de identificar qué versión genera **mayor número de usuarios convertidos**.

### Prueba z test

**Hipótesis:**
- **Hipótesis nula (H₀):** La tasa de conversión es igual en ambas páginas
- **Hipótesis alternativa (H₁):** la tasa de conversión es diferente entre las páginas

In [ ]:
# Número de usuarios convertidos por página
conversion = df.groupby('landing')['converted'].sum()

# Total de usuarios por página
total= df.groupby('landing')['converted'].count()
print("Usuarios convertidos por página:\n", conversion)
print("\nTotal de usuarios por página:\n", total)


In [ ]:
exitos= [conversion['A'], conversion['B']]
observaciones= [total['A'], total['B']]

In [ ]:
# Aplicar prueba
z_stat, p_value= proportions_ztest (exitos, observaciones)


# Visualizar resultados
print(f"Estadístico : {z_stat}")
print(f"Valor p: {p_value}")

alpha = 0.05
if p_value < alpha:
    print('Rechazamos la hipótesis nula, hay evidencia de una diferencia.')
else:
    print('No rechazamos la hipótesis nula, no hay evidencia de una diferencia.')


In [ ]:
tasa_A = exitos[0]/observaciones[0]
tasa_B = exitos[1]/observaciones[1]
if tasa_A > tasa_B:
    print(f"\nLa página A tiene una tasa de conversión mayor: {tasa_A:.2%}")
elif tasa_B > tasa_A:
    print(f"\nLa página B tiene una tasa de conversión mayor: {tasa_B:.2%}")
else:
    print('ambas páginas tienen la misma conversión')

### 📝 Conclusión e interpretación

**Decisión:**  
'Rechazamos la hipótesis nula, hay evidencia de una diferencia estadísticamente significativa de que la tasa de conversion es diferente entre la página A y B.

**Interpretación de negocio:**  
La página B tienen una mayor tasa de conversión(15.96%)comparada con la página A.
Este resultado se basa en una muestra de usuarios y no evalúa si la magnitud de la diferencia es suficiente para justificar una decisión de negocio.

## 🔗 Paso 4: Revisar la relación entre la fuente de tráfico y la conversión

Se analiza si existe una **asociación estadísticamente significativa** entre la **fuente de tráfico** (`traffic_source`) y la **conversión** (`converted`), para identificar qué canales generan más conversiones.

### Prueba: chi cuadrado

**Hipótesis:**
- **Hipótesis nula (H₀):** no existe una relación estadísticamente significativa entre las variables
- **Hipótesis alternativa (H₁):** existe una relación estadísticamente significativa entre las variables

In [ ]:
tabla = pd.crosstab(df['traffic_source'], df['converted'])

In [ ]:
# Aplicar prueba
chi2_stat, p_value, dof, expected = chi2_contingency(tabla)
print(f"Estadístico chi2: {chi2_stat:.3f}")
print(f"p_value: {p_value:.3f}")
print(f"Degrees of freedom: {dof}")
print("\nFrecuencias esperadas:")
print(expected)


In [ ]:
alpha = 0.05
if p_value < alpha:
    print('Rechazamos la hipótesis nula, hay evidencia significativa de relación entre las variables')
else:
    print('No rechazamos la hipótesis nula, no hay evidencia significativa de relación entre las variables')

In [ ]:
pd.crosstab(df['traffic_source'], df['converted'], normalize='index')*100

### 📝 Conclusión e interpretación

**Decisión:**  
Rechazamos la hipótesis nula, hay evidencia estadísticamente significativa de relación entre las variables

**Interpretación de negocio:**  
Con base en la prueba chi2 encontramos que el canal del usuario está relacionado con la conversión.
los usuarios de Email presentan una tasa de conversión mayor a los demás canales.
Este resultado se basa en una muestra de usuarios y no implica causalidad directa ni evalúa si esta diferencia justifica decisiones diferenciadas desde una perspectiva de negocio.


## 👤 Paso 5: Revisar la relación entre el tipo de usuario y la conversión

Se analiza si existe una **asociación estadísticamente significativa** entre el **tipo de usuario** (`user_type`) y la **conversión** (`converted`), entendiendo que un usuario recurrente puede haber visitado antes sin necesariamente convertirse en cliente en esta ocasión.

El objetivo es identificar qué perfiles muestran mayor probabilidad de conversión dentro del contexto analizado.

### Prueba chi cuadrado

**Hipótesis:**
- **Hipótesis nula (H₀):** no existe una relación estadísticamente significativa entre las variables
- **Hipótesis alternativa (H₁):** existe una relación estadísticamente significativa entre las variables

In [ ]:
tabla2 = pd.crosstab(df['user_type'], df['converted'])

In [ ]:
chi2_stat, p_value, dof, expected = chi2_contingency(tabla2)
print(f"Estadístico chi2: {chi2_stat:.3f}")
print(f"p_value: {p_value:.3f}")
print(f"Degrees of freedom: {dof}")
print("\nFrecuencias esperadas:")
print(expected)

In [ ]:

alpha = 0.05
if p_value < alpha:
    print('Rechazamos la hipótesis nula, hay evidencia significativa de relación entre las variables')
else:
    print('No rechazamos la hipótesis nula, no hay evidencia significativa de relación entre las variables')


In [ ]:
pd.crosstab(df['user_type'], df['converted'], normalize='index')*100

### 📝 Conclusión e interpretación

**Decisión:**  
No rechazamos la hipótesis nula, no hay evidencia significativa de relación entre las variables

**Interpretación de negocio:**  
Con base en la prueba chi2 encontramos que el tipo de usuario no está relacionado con la conversión. los usuarios nuevos presentan una tasa de conversión mayor a los usuarios recurren. Este resultado se basa en una muestra de usuarios y no implica causalidad directa ni evalúa si esta diferencia justifica decisiones diferenciadas desde una perspectiva de negocio.

## 📊 Paso 6: Visualizar los resultados de variables categóricas

Se explora visualmente la relación entre variables categóricas (`traffic_source` y `user_type`) y la conversión, mostrando para cada categoría:
- la cantidad absoluta de usuarios que convirtieron y no convirtieron,
- la proporción de usuarios que convirtieron y no convirtieron.

Esto permite analizar tanto el impacto en volumen como la efectividad relativa de cada categoría y reforzar los resultados obtenidos en las pruebas estadísticas.

### Relación entre la fuente de tráfico y la conversión

In [ ]:

ax = sns.countplot(data=df, x='traffic_source', hue='converted')

for bar in ax.patches:
    height = bar.get_height()
    ax.text(
        x=bar.get_x() + bar.get_width() / 2,
        y=height,
        s=height,
        ha='center',
        va='bottom',
        fontsize=11
    )

plt.title('Conversiones por fuente de tráfico', fontsize=15)
plt.xlabel('Fuente de tráfico', fontsize=12)
plt.ylabel('Cantidad de usuarios', fontsize=12)
plt.ylim(0, 19000)
plt.legend(title='¿Convirtió?', labels=['No (0)', 'Sí (1)'])
plt.show()


Este gráfico muestra la cantidad absoluta de usuarios que convirtieron por cada diferente fuente de tráfico

In [ ]:
tabla= pd.crosstab(df['traffic_source'], df['converted'], normalize='index')*100
tabla

In [ ]:


ax= tabla.plot(kind ='bar', stacked = True, figsize=(10,6))
plt.title('Proporción de conversiones', fontsize=15)
plt.xlabel('Fuente del tráfico', fontsize= 12)
plt.ylabel('Cantidad de usuarios', fontsize= 12)
ax.set_xlim(-1,4)
plt.legend(title='Convirtió?', labels=['No (0)', 'Sí (1)'], loc='upper right')
plt.show()



 Este gráfico muestra la proporción de usuarios que convirtieron por cada diferente fuente de tráfico

### Relación entre el tipo de usuario y la conversión

In [ ]:
ax = sns.countplot(data=df, x='user_type', hue='converted')

for bar in ax.patches:
    height = bar.get_height()
    ax.text(
        x=bar.get_x() + bar.get_width() / 2,
        y=height,
        s=height,
        ha='center',
        va='bottom',
        fontsize=11
    )

plt.title('Conversiones por tipo de usuario', fontsize=15)
plt.xlabel('tipo de usuario', fontsize=12)
plt.ylabel('Cantidad de usuarios', fontsize=12)
plt.ylim(0, 25000)
plt.legend(title='¿Convirtió?', labels=['No (0)', 'Sí (1)'])
plt.show()

Este gráfico muestra la cantidad absoluta de usuarios que convirtieron por cada tipo de usuario.

In [ ]:
tabla3= pd.crosstab(df['user_type'], df['converted'], normalize='index')*100
tabla3

In [ ]:
ax= tabla3.plot(kind ='bar', stacked = True, figsize=(10,6))
plt.title('Proporción de usuarios', fontsize=15)
plt.xlabel('Tipo de usuario', fontsize= 12)
plt.ylabel('Cantidad de usuarios', fontsize= 12)
ax.set_xlim(-1,4)
plt.legend(title='Convirtió?', labels=['No (0)', 'Sí (1)'], loc='upper right')
plt.show()



 Este gráfico muestra la proporción de usuarios que convirtieron por cada tipo de usuario

## 🧩 Paso 7. Insight Ejecutivo para Stakeholders

Se traducen los hallazgos del análisis del experimento A/B en conclusiones accionables para el negocio, enfocadas en **versión de página, conversión, gasto promedio, canales de tráfico y tipo de usuario**.

**Preguntas a responder:**  
- ¿Qué página genera mayor conversión y gasto promedio?  
- ¿Qué canales de tráfico son más efectivos para generar conversiones?  
- ¿Existen diferencias significativas según el tipo de usuario?  
- ¿Qué recomendaciones se pueden tomar para optimizar la estrategia de marketing?


---


### 🌟 Insight Ejecutivo basado en el Experimento A/B

#### 🔍 **Comparación de página (A vs B)**  

**Gasto promedio por usuario que convirtió:**
- Página A: 61.08
- Página B: 68.74
- **Interpretación:**
con base en la evidencia arrojada por la prueba t de student, se mostró que la página B tiene un gasto promedio mas grande que la página A. Este resultado se basa en una muestra de usuarios que realizaron una compra y no evalúa si la diferencia observada es suficiente para justificar un cambio desde una perspectiva de negocio.
<br>

**Tasa de conversión:**
- Página A: 0.12%
- Página B: 15.96%  
- **Interpretación:**
Con base en la evidencia arrojada por la prueba z test, se mostró que la tasa de conversión en la página B es mayor que en la Página A. Este resultado se basa en una muestra de usuarios que realizaron una compra y no evalúa si la diferencia observada es suficiente para justificar un cambio desde una perspectiva de negocio.

#### 📊 **Segmentación por fuente de tráfico**
- Existe evidencia estadísticamente significativa de relación entre las variables fuente de tráfico y relación.
- **Interpretación:**
 Con base en la prueba chi2 encontramos que el canal del usuario está relacionado con la conversión. los usuarios de Email presentan una tasa de conversión mayor a los demás canales. Este resultado se basa en una muestra de usuarios y no implica causalidad directa ni evalúa si esta diferencia justifica decisiones diferenciadas desde una perspectiva de negocio.
 ---

#### 📊 **Segmentación por tipo de usuario**
- No hay evidencia estadísticamente significativa de relación entre las variables de tipo de usuario y conversión.
- **Interpretación:**
Con base en la prueba chi2 encontramos que el tipo de usuario no está relacionado con la conversión. los usuarios nuevos presentan una tasa de conversión mayor a los usuarios recurren. Este resultado se basa en una muestra de usuarios y no implica causalidad directa ni evalúa si esta diferencia justifica decisiones diferenciadas desde una perspectiva de negocio.
---

Las visualizaciones usadas respaldan los resultados estadísticos de pasos anteriores.
Fuente de tráfico-conversión
-Gráfico de barras
-Gráfico de barras apilado

Tipo de usuario-conversión
Gráfico de barras
-Gráfico de barras apilado
---

#### 💡 **Recomendaciones de negocio:**

-Implementar la versión B de la landing page: La página B presentó una tasa de conversión significativamente mayor que la página A (15.96%) y también mostró un mayor gasto promedio entre los usuarios que realizaron una conversión (68.75 vs. 61.09). Por ello, se recomienda utilizar la versión B como página principal y monitorear su desempeño después de la implementación para confirmar que los resultados se mantengan.

-Fortalecer el canal Email: El análisis mostró una relación estadísticamente significativa entre la fuente de tráfico y la conversión, y Email presentó la mayor tasa de conversión entre los canales analizados (14.99%). Se recomienda reforzar las campañas de Email y analizar qué contenidos, promociones o segmentos generan mejores resultados, sin descuidar los demás canales.